<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes/blob/main/Modulo_de_funciones_requeridas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
matriz_carga.py

"""

from copy import deepcopy
from typing import List, Tuple, Dict, Any


Matriz = List[List[float]]


# ---------------------------------------------------------------------------
# 1. Módulo de Validación y Coherencia Dimensional
# ---------------------------------------------------------------------------
def validar_matrices(cargas: Matriz, capacidades: Matriz) -> bool:

    if not isinstance(cargas, list) or not isinstance(capacidades, list):
        return False

    n_cargas = len(cargas)
    n_capacidades = len(capacidades)

    if n_cargas < 2 or n_capacidades < 2:
        return False
    if n_cargas != n_capacidades:
        return False

    # Verificar que 'cargas' sea una matriz regular (todas las filas iguales)
    m_cargas = None
    for fila in cargas:
        if not isinstance(fila, list):
            return False
        if m_cargas is None:
            m_cargas = len(fila)
        elif len(fila) != m_cargas:
            return False

    # Verificar que 'capacidades' sea una matriz regular
    m_capacidades = None
    for fila in capacidades:
        if not isinstance(fila, list):
            return False
        if m_capacidades is None:
            m_capacidades = len(fila)
        elif len(fila) != m_capacidades:
            return False

    if m_cargas is None or m_capacidades is None:
        return False
    if m_cargas < 2 or m_capacidades < 2:
        return False
    if m_cargas != m_capacidades:
        return False

    n, m = n_cargas, m_cargas

    # Verificar rangos de valores celda por celda
    for i in range(n):
        for j in range(m):
            peso = cargas[i][j]
            capacidad = capacidades[i][j]

            if isinstance(peso, bool) or isinstance(capacidad, bool):
                return False
            if not isinstance(peso, (int, float)):
                return False
            if not isinstance(capacidad, (int, float)):
                return False
            if peso < 0:
                return False
            if capacidad <= 0:
                return False

    return True


# ---------------------------------------------------------------------------
# 2. Módulo de Cálculo de Ocupación y Detección de Sobrecarga
# ---------------------------------------------------------------------------
def calcular_ocupacion(cargas: Matriz, capacidades: Matriz) -> Dict[str, Any]:

    if not validar_matrices(cargas, capacidades):
        raise ValueError("Las matrices de cargas y capacidades no son válidas.")

    n = len(cargas)
    m = len(cargas[0])

    porcentajes: Matriz = [[0.0 for _ in range(m)] for _ in range(n)]
    sobrecargas: List[Tuple[int, int]] = []

    for i in range(n):
        for j in range(m):
            porcentaje = (cargas[i][j] / capacidades[i][j]) * 100.0
            porcentajes[i][j] = porcentaje
            if porcentaje > 100.0:
                sobrecargas.append((i, j))

    return {
        "porcentajes": porcentajes,
        "sobrecargas": sobrecargas,
    }


# ---------------------------------------------------------------------------
# 3. Módulo de Evaluación de Balance y Simetría
# ---------------------------------------------------------------------------
def evaluar_balance(cargas: Matriz, tolerancia: float) -> Dict[str, Any]:

    if not isinstance(cargas, list) or len(cargas) < 1:
        raise ValueError("La matriz de cargas debe tener al menos 1 fila.")

    m = len(cargas[0]) if cargas else 0
    for fila in cargas:
        if not isinstance(fila, list) or len(fila) != m:
            raise ValueError("La matriz de cargas debe ser regular (filas de igual longitud).")
    if m < 2:
        raise ValueError("La matriz de cargas debe tener al menos 2 columnas.")

    n = len(cargas)

    # Peso total por fila (longitudinal)
    pesos_fila: List[float] = [sum(fila) for fila in cargas]

    # Partición de columnas para el balance lateral (izquierda/derecha)
    if m % 2 == 0:
        columnas_izquierda = range(0, m // 2)
        columnas_derecha = range(m // 2, m)
    else:
        centro = m // 2
        columnas_izquierda = range(0, centro)
        columnas_derecha = range(centro + 1, m)

    suma_izquierda = sum(cargas[i][j] for i in range(n) for j in columnas_izquierda)
    suma_derecha = sum(cargas[i][j] for i in range(n) for j in columnas_derecha)

    desbalance_lateral = abs(suma_izquierda - suma_derecha)
    balance_ok = desbalance_lateral <= tolerancia

    return {
        "pesos_fila": pesos_fila,
        "desbalance_lateral": desbalance_lateral,
        "balance_ok": balance_ok,
    }


# ---------------------------------------------------------------------------
# 4. Módulo de Extracción de Submatriz de Sobrecarga Crítica
# ---------------------------------------------------------------------------
def extraer_submatriz_critica(porcentajes: Matriz, k: int, p: int) -> Matriz:

    if not isinstance(porcentajes, list) or len(porcentajes) == 0:
        raise ValueError("La matriz de porcentajes no puede estar vacía.")

    n = len(porcentajes)
    m = len(porcentajes[0])
    for fila in porcentajes:
        if len(fila) != m:
            raise ValueError("La matriz de porcentajes debe ser regular.")

    if not isinstance(k, int) or not isinstance(p, int) or k <= 0 or p <= 0:
        raise ValueError("k y p deben ser enteros positivos.")
    if k > n or p > m:
        raise ValueError(
            f"La ventana {k}x{p} excede las dimensiones de la matriz ({n}x{m})."
        )

    mejor_promedio = float("-inf")
    mejor_conteo_sobrecarga = -1
    mejor_submatriz: Matriz = []

    for i in range(n - k + 1):
        for j in range(m - p + 1):
            ventana = [fila[j:j + p] for fila in porcentajes[i:i + k]]
            valores = [valor for fila in ventana for valor in fila]

            promedio = sum(valores) / len(valores)
            conteo_sobrecarga = sum(1 for v in valores if v > 100.0)

            es_mejor = (
                promedio > mejor_promedio
                or (promedio == mejor_promedio and conteo_sobrecarga > mejor_conteo_sobrecarga)
            )
            if es_mejor:
                mejor_promedio = promedio
                mejor_conteo_sobrecarga = conteo_sobrecarga
                mejor_submatriz = deepcopy(ventana)

    return mejor_submatriz